In [23]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [24]:
corpus = [
    "AI is teaching machines to think like humans.",
    "Machine learning learns from data, not rules.",
    "Data is the fuel of artificial intelligence.",
    "Algorithms turn data into decisions.",
    "AI is transforming every industry.",
    "Learning models improve with experience.",
    "Deep learning mimics the human brain.",
    "Neural networks process complex patterns.",
    "AI can automate repetitive tasks.",
    "Machine learning predicts future outcomes.",
    "Good data leads to better models.",
    "AI is shaping the future of technology.",
    "Bias in data leads to biased AI.",
    "Training is the heart of machine learning.",
    "AI systems learn through iteration.",
    "Feature extraction improves model accuracy.",
    "AI can analyze large datasets quickly.",
    "Machine learning is everywhere today.",
    "Ethical AI is more important than ever.",
    "Supervised learning uses labeled data.",
    "Artificial intelligence enhances decision making.",
    "Machine learning models require quality data.",
    "AI systems can learn patterns from examples.",
    "Big data drives modern AI applications.",
    "Automation through AI saves time and effort.",
    "AI helps in solving complex real world problems.",
    "Neural networks consist of multiple layers.",
    "Deep learning improves accuracy in predictions.",
    "AI is widely used in healthcare and finance.",
    "Training data determines model performance.",
    "AI can recognize speech and images.",
    "Machine learning algorithms evolve over time.",
    "Data preprocessing is essential for good results.",
    "AI assists in recommendation systems.",
    "Model evaluation ensures reliable predictions."
]

print("Number of sentences:", len(corpus))
print("\nSample corpus:")
for line in corpus:
    print("-", line)

Number of sentences: 35

Sample corpus:
- AI is teaching machines to think like humans.
- Machine learning learns from data, not rules.
- Data is the fuel of artificial intelligence.
- Algorithms turn data into decisions.
- AI is transforming every industry.
- Learning models improve with experience.
- Deep learning mimics the human brain.
- Neural networks process complex patterns.
- AI can automate repetitive tasks.
- Machine learning predicts future outcomes.
- Good data leads to better models.
- AI is shaping the future of technology.
- Bias in data leads to biased AI.
- Training is the heart of machine learning.
- AI systems learn through iteration.
- Feature extraction improves model accuracy.
- AI can analyze large datasets quickly.
- Machine learning is everywhere today.
- Ethical AI is more important than ever.
- Supervised learning uses labeled data.
- Artificial intelligence enhances decision making.
- Machine learning models require quality data.
- AI systems can learn patt

In [25]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

word_index = tokenizer.word_index
total_words = len(word_index) + 1

print("Vocabulary size:", total_words)
print("\nWord index:")
print(word_index)

Vocabulary size: 125

Word index:
{'ai': 1, 'learning': 2, 'data': 3, 'is': 4, 'machine': 5, 'in': 6, 'the': 7, 'of': 8, 'can': 9, 'to': 10, 'models': 11, 'systems': 12, 'model': 13, 'and': 14, 'from': 15, 'artificial': 16, 'intelligence': 17, 'algorithms': 18, 'deep': 19, 'neural': 20, 'networks': 21, 'complex': 22, 'patterns': 23, 'future': 24, 'good': 25, 'leads': 26, 'training': 27, 'learn': 28, 'through': 29, 'improves': 30, 'accuracy': 31, 'time': 32, 'predictions': 33, 'teaching': 34, 'machines': 35, 'think': 36, 'like': 37, 'humans': 38, 'learns': 39, 'not': 40, 'rules': 41, 'fuel': 42, 'turn': 43, 'into': 44, 'decisions': 45, 'transforming': 46, 'every': 47, 'industry': 48, 'improve': 49, 'with': 50, 'experience': 51, 'mimics': 52, 'human': 53, 'brain': 54, 'process': 55, 'automate': 56, 'repetitive': 57, 'tasks': 58, 'predicts': 59, 'outcomes': 60, 'better': 61, 'shaping': 62, 'technology': 63, 'bias': 64, 'biased': 65, 'heart': 66, 'iteration': 67, 'feature': 68, 'extraction

In [26]:
input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total training sequences:", len(input_sequences))
print("\nSome sequences before padding:")
for seq in input_sequences[:10]:
    print(seq)

Total training sequences: 176

Some sequences before padding:
[1, 4]
[1, 4, 34]
[1, 4, 34, 35]
[1, 4, 34, 35, 10]
[1, 4, 34, 35, 10, 36]
[1, 4, 34, 35, 10, 36, 37]
[1, 4, 34, 35, 10, 36, 37, 38]
[5, 2]
[5, 2, 39]
[5, 2, 39, 15]


In [27]:
max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

print("Maximum sequence length:", max_seq_len)
print("\nPadded sequences:")
print(input_sequences[:10])

Maximum sequence length: 8

Padded sequences:
[[ 0  0  0  0  0  0  1  4]
 [ 0  0  0  0  0  1  4 34]
 [ 0  0  0  0  1  4 34 35]
 [ 0  0  0  1  4 34 35 10]
 [ 0  0  1  4 34 35 10 36]
 [ 0  1  4 34 35 10 36 37]
 [ 1  4 34 35 10 36 37 38]
 [ 0  0  0  0  0  0  5  2]
 [ 0  0  0  0  0  5  2 39]
 [ 0  0  0  0  5  2 39 15]]


In [28]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (176, 7)
Shape of y: (176, 125)


In [29]:
model = Sequential([
    Embedding(input_dim=total_words, output_dim=50, input_length=max_seq_len - 1),
    LSTM(150, return_sequences=True),
    LSTM(100),
    Dense(100, activation='relu'),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='loss', patience=20)

history = model.fit(X, y, epochs=500,  verbose=1)

Epoch 1/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.0284 - loss: 4.8278
Epoch 2/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0568 - loss: 4.8177
Epoch 3/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0511 - loss: 4.7823
Epoch 4/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0511 - loss: 4.6806
Epoch 5/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0511 - loss: 4.6246
Epoch 6/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0511 - loss: 4.5784
Epoch 7/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0511 - loss: 4.5376
Epoch 8/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0511 - loss: 4.4971
Epoch 9/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0511 - loss: 4.4618
Epoch 10/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0682 - loss: 4.4249
Epoch 11/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0398 - loss: 4.3892
Epoch 12/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0398 - lo

In [31]:
def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text

In [32]:
print(generate_text("deep learning", 2))
print(generate_text("machine learning", 2))
print(generate_text("python is", 2))

deep learning improves accuracy
machine learning predicts future
python is is everywhere


In [33]:
print(generate_text("artificial intelligence", 2))
print(generate_text("data science", 2))
print(generate_text("lstm is", 3))

artificial intelligence enhances decision
data science is the
lstm is is everywhere today
